In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler

In [7]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [8]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [9]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [10]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [11]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [12]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [13]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [14]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [15]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [16]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [17]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# Set y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [18]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [19]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [20]:
# Change the name of a column 'DFS_event' in the clincial_test 
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [21]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

## Feature Selection

### COX PLSR 

In [22]:
# Choose features from the result of Cox PLSR in R
selected_features = [
"hpv_related",
"uicc8_III-IV",
"cavum_oris",
"shape_MajorAxisLength",
"female",
"shape_Elongation",
"shape_Sphericity"
] 

In [23]:
X_plsr = X.loc[:, selected_features]
X_new = X_plsr.copy()

In [24]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, plsr]

# Standardization

In [25]:
# Copy the original X for later 
original_X = X.copy()

In [26]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [27]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [28]:
X_new

,hpv_related,uicc8_III-IV,cavum_oris,shape_MajorAxisLength,female,shape_Elongation,shape_Sphericity
0,0.0,0.0,0,42.073251,1,0.600926,0.761164
1,0.0,0.0,0,24.613845,0,0.841579,0.697049
2,0.0,1.0,1,48.030294,0,0.772821,0.565792
3,0.0,0.0,0,25.589900,0,0.847727,0.684364
4,0.0,0.0,0,34.684750,0,0.831483,0.503142
...,...,...,...,...,...,...,...
134,1.0,0.0,0,33.069705,0,0.680294,0.742102
135,1.0,1.0,0,41.043692,0,0.758193,0.722918
136,1.0,0.0,0,36.618802,0,0.770113,0.652963
137,1.0,1.0,0,45.870392,0,0.628897,0.724255


In [29]:
X_new_std

,hpv_related,uicc8_III-IV,cavum_oris,shape_MajorAxisLength,female,shape_Elongation,shape_Sphericity
0,0.0,0.0,0,-0.098422,1,-0.681983,1.075042
1,0.0,0.0,0,-1.209119,0,1.006304,0.242767
2,0.0,1.0,1,0.280542,0,0.523938,-1.461057
3,0.0,0.0,0,-1.147026,0,1.049432,0.078112
4,0.0,0.0,0,-0.568449,0,0.935477,-2.274305
...,...,...,...,...,...,...,...
134,1.0,0.0,0,-0.671191,0,-0.125182,0.827589
135,1.0,1.0,0,-0.163918,0,0.421312,0.578577
136,1.0,0.0,0,-0.445412,0,0.504939,-0.329498
137,1.0,1.0,0,0.143137,0,-0.485754,0.595927


In [30]:
MAASTRO_new 

,hpv_related,uicc8_III-IV,cavum_oris,shape_MajorAxisLength,female,shape_Elongation,shape_Sphericity
0,1,0,0,50.002093,0,0.765178,0.668072
1,0,1,0,41.753334,0,0.776540,0.669961
2,0,1,0,44.375483,0,0.697164,0.624081
3,0,1,0,46.115989,1,0.574636,0.577624
4,1,0,0,54.394967,0,0.633419,0.630933
...,...,...,...,...,...,...,...
94,0,1,0,34.218615,1,0.882411,0.671754
95,0,1,0,51.046869,0,0.535802,0.632189
96,1,1,0,50.417953,0,0.716610,0.645548
97,1,0,0,44.901412,0,0.665145,0.727488


In [31]:
MAASTRO_new_std

,hpv_related,uicc8_III-IV,cavum_oris,shape_MajorAxisLength,female,shape_Elongation,shape_Sphericity
0,1,0,0,0.405980,0,0.470316,-0.133381
1,0,1,0,-0.118774,0,0.550025,-0.108854
2,0,1,0,0.048037,0,-0.006831,-0.704414
3,0,1,0,0.158761,1,-0.866418,-1.307469
4,1,0,0,0.685437,0,-0.454030,-0.615466
...,...,...,...,...,...,...,...
94,0,1,0,-0.598102,1,1.292755,-0.085581
95,0,1,0,0.472444,0,-1.138850,-0.599161
96,1,1,0,0.432435,0,0.129595,-0.425762
97,1,0,0,0.081495,0,-0.231458,0.637891


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [32]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 20:17:45,871] A new study created in memory with name: no-name-6b8808d0-c28f-4ba8-927d-cd81bcc9d0ca
python(31267) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-16 20:17:49,649] A new study created in memory with name: no-name-45b4b325-7361-4e45-8653-a4aa8845b7e7


Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7319148936170212
Fold 4 C-index: 0.7718631178707225
Fold 5 C-index: 0.7253218884120172
[I 2024-04-16 20:17:49,641] Trial 0 finished with value: 0.7095063198916233 and parameters: {}. Best is trial 0 with value: 0.7095063198916233.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7095063198916233], datetime_start=datetime.datetime(2024, 4, 16, 20, 17, 46, 3909), datetime_complete=datetime.datetime(2024, 4, 16, 20, 17, 49, 640700), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7095063198916233


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24547621902232983
Fold 2 IBS: 0.18646691736393076
Fold 3 IBS: 0.19716544700257893
Fold 4 IBS: 0.19436890362488465
Fold 5 IBS: 0.18028846636930945
[I 2024-04-16 20:17:49,893] Trial 0 finished with value: 0.20075319067660674 and parameters: {}. Best is trial 0 with value: 0.20075319067660674.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20075319067660674], datetime_start=datetime.datetime(2024, 4, 16, 20, 17, 49, 682222), datetime_complete=datetime.datetime(2024, 4, 16, 20, 17, 49, 893241), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20075319067660674


In [33]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [34]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.71
train_ibs:  0.201


#### Test

In [35]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [36]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.565
IBS score: 0.265


In [37]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [38]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [39]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:17:50,075] A new study created in memory with name: no-name-f7782328-b24a-4dc5-8554-aaf610609497


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5916334661354582
Fold 2 C-index: 0.7286821705426356


[I 2024-04-16 20:17:50,258] A new study created in memory with name: no-name-5f576437-c7be-4862-8d7b-2bbe012a6ec8


Fold 3 C-index: 0.5702127659574469
Fold 4 C-index: 0.7167300380228137
Fold 5 C-index: 0.6695278969957081
[I 2024-04-16 20:17:50,254] Trial 0 finished with value: 0.6553572675308125 and parameters: {}. Best is trial 0 with value: 0.6553572675308125.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6553572675308125], datetime_start=datetime.datetime(2024, 4, 16, 20, 17, 50, 111454), datetime_complete=datetime.datetime(2024, 4, 16, 20, 17, 50, 254369), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6553572675308125


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709428728842
Fold 2 IBS: 0.23203987166610096
Fold 3 IBS: 0.22898186706001508
Fold 4 IBS: 0.2419747601009159
Fold 5 IBS: 0.22939558449397743
[I 2024-04-16 20:17:50,448] Trial 0 finished with value: 0.23592783552165955 and parameters: {}. Best is trial 0 with value: 0.23592783552165955.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592783552165955], datetime_start=datetime.datetime(2024, 4, 16, 20, 17, 50, 294589), datetime_complete=datetime.datetime(2024, 4, 16, 20, 17, 50, 448409), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592783552165955


In [40]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [41]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.655
train_ibs:  0.236


#### Test

In [42]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [43]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.537


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [44]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [45]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:17:50,619] A new study created in memory with name: no-name-d2a3f5cb-508c-493e-b2e6-95b138507ff9


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7248062015503876


[I 2024-04-16 20:17:50,902] A new study created in memory with name: no-name-1639a0e6-a8bf-4129-acd2-ad0aab0e2fbf


Fold 3 C-index: 0.7319148936170212
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:17:50,896] Trial 0 finished with value: 0.7094195636674461 and parameters: {}. Best is trial 0 with value: 0.7094195636674461.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7094195636674461], datetime_start=datetime.datetime(2024, 4, 16, 20, 17, 50, 662905), datetime_complete=datetime.datetime(2024, 4, 16, 20, 17, 50, 896345), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7094195636674461


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24339534410179192
Fold 2 IBS: 0.185182554310044
Fold 3 IBS: 0.19735806201590553
Fold 4 IBS: 0.19425154567068226
Fold 5 IBS: 0.17858505650283987
[I 2024-04-16 20:17:51,197] Trial 0 finished with value: 0.19975451252025272 and parameters: {}. Best is trial 0 with value: 0.19975451252025272.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.19975451252025272], datetime_start=datetime.datetime(2024, 4, 16, 20, 17, 50, 941477), datetime_complete=datetime.datetime(2024, 4, 16, 20, 17, 51, 197070), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.19975451252025272


In [46]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [47]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.709
train_ibs:  0.2


#### Test

In [48]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [49]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.566


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.264


In [50]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [51]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:17:51,437] A new study created in memory with name: no-name-654568bd-1c78-4798-9db9-ad7fec03d660


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7319148936170212
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:17:51,701] Trial 0 finished with value: 0.7094195636674461 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7094195636674461.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7319148936170212
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:17:51,945] Trial 1 finished with value: 0.7086227509184421 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7094195636674461.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:17:52,200] Trial 2 finished with value: 0.7094738147482293 and parameters: {'l1_ratio': 0.22692876841884668}. B

Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:17:58,632] Trial 24 finished with value: 0.7094738147482293 and parameters: {'l1_ratio': 0.2193939366669308}. Best is trial 2 with value: 0.7094738147482293.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7319148936170212
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:17:58,889] Trial 25 finished with value: 0.7086227509184421 and parameters: {'l1_ratio': 0.36991167247338663}. Best is trial 2 with value: 0.7094738147482293.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7124463519313304
[I 2024-04-16 20:17:59,215] Trial 26 finished with value: 0.708615445649517 and parameters: {'l1_ratio': 0.13147141129458395}. Best is trial 2 with value: 0.7094738147482293.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7361

Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:18:05,877] Trial 48 finished with value: 0.7086227509184421 and parameters: {'l1_ratio': 0.7768768486379835}. Best is trial 2 with value: 0.7094738147482293.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:18:06,135] Trial 49 finished with value: 0.7094738147482293 and parameters: {'l1_ratio': 0.2758853961660157}. Best is trial 2 with value: 0.7094738147482293.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:18:06,356] Trial 50 finished with value: 0.7094738147482293 and parameters: {'l1_ratio': 0.12640640297302158}. Best is trial 2 with value: 0.7094738147482293.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7248

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.24353047233571218
Fold 2 IBS: 0.18505548303484637
Fold 3 IBS: 0.19730689652689593
Fold 4 IBS: 0.19422737036147178
Fold 5 IBS: 0.17851015969988163
[I 2024-04-16 20:18:18,024] Trial 0 finished with value: 0.19972607639176157 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.19972607639176157.
Fold 1 IBS: 0.24360127705755819
Fold 2 IBS: 0.185007449755265
Fold 3 IBS: 0.19721867191474393
Fold 4 IBS: 0.19415153821706493
Fold 5 IBS: 0.17850233659697254
[I 2024-04-16 20:18:18,298] Trial 1 finished with value: 0.19969625470832092 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.19969625470832092.
Fold 1 IBS: 0.24351257107264307
Fold 2 IBS: 0.18501447211139116
Fold 3 IBS: 0.19712891614318392
Fold 4 IBS: 0.1941210385253576
Fold 5 IBS: 0.17852330815931183
[I 2024-04-16 20:18:18,555] Trial 2 finished with value: 0.1996600612023775 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.1996600612023

Fold 5 IBS: 0.178598053564367
[I 2024-04-16 20:18:23,503] Trial 25 finished with value: 0.21440083145434272 and parameters: {'l1_ratio': 0.06349940013770836}. Best is trial 16 with value: 0.19960308709852081.
Fold 1 IBS: 0.24351366760725107
Fold 2 IBS: 0.18496422300962823
Fold 3 IBS: 0.19700342856991873
Fold 4 IBS: 0.1940991769035908
Fold 5 IBS: 0.17855757385038704
[I 2024-04-16 20:18:23,698] Trial 26 finished with value: 0.19962761398815518 and parameters: {'l1_ratio': 0.17279016655324594}. Best is trial 16 with value: 0.19960308709852081.
Fold 1 IBS: 0.24357562966361976
Fold 2 IBS: 0.18504134792632615
Fold 3 IBS: 0.1972939882535071
Fold 4 IBS: 0.1942054655303824
Fold 5 IBS: 0.17848191293841448
[I 2024-04-16 20:18:23,888] Trial 27 finished with value: 0.19971966886245 and parameters: {'l1_ratio': 0.58416616744615}. Best is trial 16 with value: 0.19960308709852081.
Fold 1 IBS: 0.24349097130911254
Fold 2 IBS: 0.18497216691709234
Fold 3 IBS: 0.19725200652728428
Fold 4 IBS: 0.194174704043

Fold 1 IBS: 0.24366600858696846
Fold 2 IBS: 0.1849908193564825
Fold 3 IBS: 0.19675372997116544
Fold 4 IBS: 0.19405308641009722
Fold 5 IBS: 0.17859317541288244
[I 2024-04-16 20:18:28,378] Trial 51 finished with value: 0.19961136394751922 and parameters: {'l1_ratio': 0.08432532131440844}. Best is trial 34 with value: 0.19958936354302614.
Fold 1 IBS: 0.24355149972756562
Fold 2 IBS: 0.18503496094507124
Fold 3 IBS: 0.19680134213895506
Fold 4 IBS: 0.1940676034270465
Fold 5 IBS: 0.1785753431827489
[I 2024-04-16 20:18:28,598] Trial 52 finished with value: 0.19960614988427744 and parameters: {'l1_ratio': 0.09694846336288854}. Best is trial 34 with value: 0.19958936354302614.
Fold 1 IBS: 0.24711925423645356
Fold 2 IBS: 0.23177243954176463
Fold 3 IBS: 0.22896133689875584
Fold 4 IBS: 0.24173903126704263
Fold 5 IBS: 0.2292179967456121
[I 2024-04-16 20:18:28,686] Trial 53 finished with value: 0.23576201173792577 and parameters: {'l1_ratio': 0.0020896128570487976}. Best is trial 34 with value: 0.1995

Fold 3 IBS: 0.1970173491910116
Fold 4 IBS: 0.19410597606698898
Fold 5 IBS: 0.17854867444748423
[I 2024-04-16 20:18:33,424] Trial 76 finished with value: 0.1996421173606743 and parameters: {'l1_ratio': 0.17731016579466904}. Best is trial 34 with value: 0.19958936354302614.
Fold 1 IBS: 0.2435525817207568
Fold 2 IBS: 0.22582879659192498
Fold 3 IBS: 0.22876806601582172
Fold 4 IBS: 0.23662444816242606
Fold 5 IBS: 0.17868124670242194
[I 2024-04-16 20:18:33,577] Trial 77 finished with value: 0.22269102783867029 and parameters: {'l1_ratio': 0.05625072097962939}. Best is trial 34 with value: 0.19958936354302614.
Fold 1 IBS: 0.24358639906882854
Fold 2 IBS: 0.18497307008813196
Fold 3 IBS: 0.19689045550507653
Fold 4 IBS: 0.19408913016201246
Fold 5 IBS: 0.1785773099461362
[I 2024-04-16 20:18:33,810] Trial 78 finished with value: 0.19962327295403715 and parameters: {'l1_ratio': 0.1257627141847904}. Best is trial 34 with value: 0.19958936354302614.
Fold 1 IBS: 0.24626114167923135
Fold 2 IBS: 0.229966

In [52]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [53]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.709
train_ibs:  0.2


#### Test

In [54]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [55]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.22692876841884668)

test_cindex : 0.565


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.08888945657798078)

test_ibs:  0.264


In [56]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [57]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 20:18:37,928] A new study created in memory with name: no-name-9dca24b4-2b75-41b9-9843-0397bcde30dc


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6772908366533864
Fold 2 C-index: 0.7441860465116279
Fold 3 C-index: 0.6680851063829787
Fold 4 C-index: 0.7357414448669202
Fold 5 C-index: 0.6652360515021459
[I 2024-04-16 20:18:40,404] Trial 0 finished with value: 0.6981078971834117 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6981078971834117.
Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.7927756653992395
Fold 5 C-index: 0.6909871244635193
[I 2024-04-16 20:18:42,080] Trial 1 finished with value: 0.7168121053676654 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 5 C-index: 0.759656652360515
[I 2024-04-16 20:19:00,606] Trial 15 finished with value: 0.7542609019301453 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 2, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 92, 'oob_score': True, 'max_samples': 0.792913180904143, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.09179883523732954, 'warm_start': True}. Best is trial 14 with value: 0.780987764930577.
Fold 1 C-index: 0.5517928286852589
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.8022813688212928
Fold 5 C-index: 0.723175965665236
[I 2024-04-16 20:19:00,743] Trial 16 finished with value: 0.7212623367742224 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 2, 'min_samples_leaf': 1, 'max_depth': 1, 'n_estimators': 5, 'oob_score': True, 'max_samples': 0.44827686549014745, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.07731335546121543, 'warm_start': True}. Best is trial 14 with value: 0.780987764930577.
Fol

Fold 1 C-index: 0.6533864541832669
Fold 2 C-index: 0.7868217054263565
Fold 3 C-index: 0.825531914893617
Fold 4 C-index: 0.8612167300380228
Fold 5 C-index: 0.8154506437768241
[I 2024-04-16 20:19:10,232] Trial 30 finished with value: 0.7884814896636174 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 108, 'oob_score': True, 'max_samples': 0.6618240078421601, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.08976817948926405, 'warm_start': True}. Best is trial 27 with value: 0.7886806158154684.
Fold 1 C-index: 0.6533864541832669
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.825531914893617
Fold 4 C-index: 0.8498098859315589
Fold 5 C-index: 0.8240343347639485
[I 2024-04-16 20:19:10,788] Trial 31 finished with value: 0.7886920528381992 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 112, 'oob_score': True, 'max_samples': 0.6801708526773039, '

...

Fold 1 IBS: 0.21696087705123832
Fold 2 IBS: 0.18051846494326523
Fold 3 IBS: 0.19858268230981502
Fold 4 IBS: 0.19548842415207185
Fold 5 IBS: 0.1939615595663194
[I 2024-04-16 20:20:45,275] Trial 32 finished with value: 0.19710240160454198 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 13, 'min_samples_leaf': 5, 'max_depth': 3, 'n_estimators': 242, 'oob_score': True, 'max_samples': 0.24335177937454738, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0403634329485552}. Best is trial 28 with value: 0.1957081731952507.
Fold 1 IBS: 0.22607044337199272
Fold 2 IBS: 0.19854056368445666
Fold 3 IBS: 0.2066706312720571
Fold 4 IBS: 0.2127158757656954
Fold 5 IBS: 0.20132348644962628
[I 2024-04-16 20:20:47,276] Trial 33 finished with value: 0.2090642001087656 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 1, 'n_estimators': 308, 'oob_score': True, 'max_samples': 0.2205178594196074, 'max_features': 'sqrt', 'min_weight_fraction_leaf'

Fold 1 IBS: 0.2227666955281551
Fold 2 IBS: 0.19165948973828925
Fold 3 IBS: 0.19545544746225493
Fold 4 IBS: 0.2051264434970915
Fold 5 IBS: 0.1988943998981424
[I 2024-04-16 20:21:14,403] Trial 48 finished with value: 0.20278049522478664 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 1, 'n_estimators': 76, 'oob_score': True, 'max_samples': 0.3886019184666001, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.06837241646154979}. Best is trial 28 with value: 0.1957081731952507.
Fold 1 IBS: 0.2329000315417291
Fold 2 IBS: 0.21266089545946018
Fold 3 IBS: 0.21523301311412954
Fold 4 IBS: 0.22365785685511483
Fold 5 IBS: 0.2105763226381107
[I 2024-04-16 20:21:15,765] Trial 49 finished with value: 0.21900562392170891 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 7, 'min_samples_leaf': 20, 'max_depth': 5, 'n_estimators': 187, 'oob_score': True, 'max_samples': 0.4696154766839547, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.22424195411650993
Fold 2 IBS: 0.1733452077865647
Fold 3 IBS: 0.1987495278992841
Fold 4 IBS: 0.18771842316860926
Fold 5 IBS: 0.19559796363556234
[I 2024-04-16 20:21:48,179] Trial 64 finished with value: 0.1959306153213061 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 10, 'min_samples_leaf': 7, 'max_depth': 13, 'n_estimators': 224, 'oob_score': True, 'max_samples': 0.6854054895166218, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0574508751550857}. Best is trial 62 with value: 0.19553556624509869.
Fold 1 IBS: 0.21880885831421684
Fold 2 IBS: 0.1800713612968455
Fold 3 IBS: 0.19937666972389528
Fold 4 IBS: 0.1948820181683657
Fold 5 IBS: 0.19924258390235944
[I 2024-04-16 20:21:51,414] Trial 65 finished with value: 0.19847629828113655 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 13, 'n_estimators': 282, 'oob_score': True, 'max_samples': 0.6872539783987465, 'max_features': 'sqrt', 'min_weight_fraction_leaf

Fold 5 IBS: 0.19489331507182253
[I 2024-04-16 20:22:20,221] Trial 79 finished with value: 0.1962283184725394 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 11, 'min_samples_leaf': 7, 'max_depth': 15, 'n_estimators': 197, 'oob_score': True, 'max_samples': 0.7204334120135304, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.06182087800738387}. Best is trial 62 with value: 0.19553556624509869.
Fold 1 IBS: 0.2215575415781244
Fold 2 IBS: 0.17551831064016074
Fold 3 IBS: 0.2003450191188829
Fold 4 IBS: 0.1893570876676837
Fold 5 IBS: 0.19512179731350374
[I 2024-04-16 20:22:22,576] Trial 80 finished with value: 0.19637995126367108 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 11, 'min_samples_leaf': 7, 'max_depth': 15, 'n_estimators': 238, 'oob_score': True, 'max_samples': 0.660392551614857, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.03604081814743047}. Best is trial 62 with value: 0.19553556624509869.
Fold 1 IBS: 0.22275825379076
Fold 2 IBS: 0.17393253

Fold 1 IBS: 0.22327215911654924
Fold 2 IBS: 0.17181605634099717
Fold 3 IBS: 0.19891371164541044
Fold 4 IBS: 0.18858035986472774
Fold 5 IBS: 0.19936010551066077
[I 2024-04-16 20:23:35,801] Trial 95 finished with value: 0.19638847849566907 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 14, 'min_samples_leaf': 6, 'max_depth': 14, 'n_estimators': 276, 'oob_score': True, 'max_samples': 0.6710713752997685, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.03072314044967749}. Best is trial 62 with value: 0.19553556624509869.
Fold 1 IBS: 0.22416864686613788
Fold 2 IBS: 0.17494700128364413
Fold 3 IBS: 0.20002311392349784
Fold 4 IBS: 0.18561514214063585
Fold 5 IBS: 0.1983466242333223
[I 2024-04-16 20:23:39,146] Trial 96 finished with value: 0.1966201056894476 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 12, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 259, 'oob_score': True, 'max_samples': 0.7160954347987437, 'max_features': 'sqrt', 'min_weight_fraction

In [58]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [59]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.805
train_ibs:  0.196


#### Test

In [60]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [61]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=7, max_leaf_nodes=9,
                     max_samples=0.8893723590453224, min_samples_split=11,
                     min_weight_fraction_leaf=0.029301059641382382,
                     n_estimators=299, random_state=123, warm_start=True)

test_cindex:  0.579


RandomSurvivalForest(max_depth=3, max_leaf_nodes=11,
                     max_samples=0.689383772643782, min_samples_leaf=7,
                     min_samples_split=14,
                     min_weight_fraction_leaf=0.04852874808590733,
                     n_estimators=230, oob_score=True, random_state=123)

test_ibs:  0.247


In [62]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [63]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [64]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:23:49,882] A new study created in memory with name: no-name-590d4d3c-fd15-410b-98ae-9e4ec2df1dfd


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.8382978723404255
Fold 4 C-index: 0.7813688212927756
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 20:23:50,567] Trial 0 finished with value: 0.7350162387199729 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7350162387199729.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:23:52,235] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.6155378486055777
Fold 2 C-index: 0.6918604651162791
Fold 3 C-index: 0.7978723404255319
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.7145922746781116
[I 2024-04-16 20:24:12,691] Trial 16 finished with value: 0.7069383652327806 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.753153713469314.
Fold 1 C-index: 0.6454183266932271
Fold 2 C-index: 0.7073643410852714
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.7547528517110266
Fold 5 C-index: 0.6738197424892703
[I 2024-04-16 20:24:13,146] Trial 17 finished with value: 0.7188242438851208 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_featur

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7790697674418605
Fold 3 C-index: 0.8851063829787233
Fold 4 C-index: 0.908745247148289
Fold 5 C-index: 0.8626609442060086
[I 2024-04-16 20:24:27,091] Trial 31 finished with value: 0.8106224444505938 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 20, 'min_samples_leaf': 2, 'max_depth': 11, 'n_estimators': 216, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.8236838576576057, 'min_weight_fraction_leaf': 0.03249179180997831}. Best is trial 31 with value: 0.8106224444505938.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.8808510638297873
Fold 4 C-index: 0.9125475285171103
Fold 5 C-index: 0.8540772532188842
[I 2024-04-16 20:24:27,569] Trial 32 finished with value: 0.8103871052445996 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 19, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 216, 'oob_score': False, 'warm_start': True, 'max_features

...

Fold 1 IBS: 0.23023776941861945
Fold 2 IBS: 0.213511288827196
Fold 3 IBS: 0.20032100930462735
Fold 4 IBS: 0.21902572357015324
Fold 5 IBS: 0.20678699212033239
[I 2024-04-16 20:26:56,982] Trial 30 finished with value: 0.2139765566481857 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 457, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.3838693489032564, 'min_weight_fraction_leaf': 0.03868084640768137}. Best is trial 12 with value: 0.20209917972851432.
Fold 1 IBS: 0.22785913525596427
Fold 2 IBS: 0.20385621569535622
Fold 3 IBS: 0.19317864058913928
Fold 4 IBS: 0.20830386583117433
Fold 5 IBS: 0.19648625224253355
[I 2024-04-16 20:27:01,884] Trial 31 finished with value: 0.20593682192283352 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 10, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0

Fold 1 IBS: 0.23166436341307534
Fold 2 IBS: 0.1970282792781475
Fold 3 IBS: 0.18846658654338827
Fold 4 IBS: 0.2040824853956187
Fold 5 IBS: 0.18804555441980503
[I 2024-04-16 20:27:53,258] Trial 45 finished with value: 0.20185745381000694 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 9, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 239, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.7884699835044351, 'min_weight_fraction_leaf': 0.02308271139521288}. Best is trial 35 with value: 0.20023521972706174.
Fold 1 IBS: 0.23259234371889384
Fold 2 IBS: 0.19671333504216923
Fold 3 IBS: 0.18808153790553353
Fold 4 IBS: 0.20560000019106228
Fold 5 IBS: 0.19051350976162718
[I 2024-04-16 20:27:55,908] Trial 46 finished with value: 0.20270014532385722 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 166, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples'

Fold 1 IBS: 0.23218744117894213
Fold 2 IBS: 0.195817634720557
Fold 3 IBS: 0.1883733812292164
Fold 4 IBS: 0.20331460709002558
Fold 5 IBS: 0.18807061354020613
[I 2024-04-16 20:28:45,253] Trial 60 finished with value: 0.20155273555178943 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 17, 'n_estimators': 332, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.8377294618227489, 'min_weight_fraction_leaf': 0.0022038644175106023}. Best is trial 35 with value: 0.20023521972706174.
Fold 1 IBS: 0.2289831783698531
Fold 2 IBS: 0.1974503679208846
Fold 3 IBS: 0.18936728070713474
Fold 4 IBS: 0.20610193104790206
Fold 5 IBS: 0.18917988658494733
[I 2024-04-16 20:28:48,805] Trial 61 finished with value: 0.20221652892614433 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 277, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples'

Fold 1 IBS: 0.23325700465373614
Fold 2 IBS: 0.18941006365861973
Fold 3 IBS: 0.19777227345442108
Fold 4 IBS: 0.20367365815974922
Fold 5 IBS: 0.19257707911996447
[I 2024-04-16 20:29:41,410] Trial 75 finished with value: 0.20333801580929817 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 14, 'min_samples_leaf': 3, 'max_depth': 13, 'n_estimators': 264, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.7937577980359053, 'min_weight_fraction_leaf': 0.03875275975124326}. Best is trial 35 with value: 0.20023521972706174.
Fold 1 IBS: 0.23017168335686908
Fold 2 IBS: 0.1910769217553898
Fold 3 IBS: 0.19558186536350308
Fold 4 IBS: 0.19877865552249777
Fold 5 IBS: 0.1933197348980885
[I 2024-04-16 20:29:44,982] Trial 76 finished with value: 0.2017857721792696 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 14, 'n_estimators': 350, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0

Fold 1 IBS: 0.23321083009190172
Fold 2 IBS: 0.19313426544526122
Fold 3 IBS: 0.1867653770716483
Fold 4 IBS: 0.19682621420661314
Fold 5 IBS: 0.1851399199104335
[I 2024-04-16 20:30:33,360] Trial 90 finished with value: 0.19901532134517158 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 16, 'n_estimators': 300, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.8499279955582367, 'min_weight_fraction_leaf': 0.022961915401987498}. Best is trial 90 with value: 0.19901532134517158.
Fold 1 IBS: 0.23178429607030182
Fold 2 IBS: 0.19405045604401192
Fold 3 IBS: 0.1863782097469378
Fold 4 IBS: 0.19756230007653738
Fold 5 IBS: 0.18523135334419819
[I 2024-04-16 20:30:36,679] Trial 91 finished with value: 0.19900132305639742 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 16, 'n_estimators': 330, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples'

In [65]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [66]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.835
train_ibs:  0.198


#### Test

In [67]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [68]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=13, max_features=None, max_leaf_nodes=19,
                   max_samples=0.8984329851688493, min_samples_leaf=1,
                   min_samples_split=4,
                   min_weight_fraction_leaf=0.0010312054923344999,
                   n_estimators=322, random_state=123, warm_start=True)

C-index score: 0.581


ExtraSurvivalTrees(max_depth=19, max_features='log2', max_leaf_nodes=10,
                   max_samples=0.8714089894245162, min_samples_leaf=1,
                   min_samples_split=11,
                   min_weight_fraction_leaf=0.025634903535808776,
                   n_estimators=334, random_state=123, warm_start=True)

IBS: 0.234


In [69]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis


#### Train

In [70]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 20:31:08,599] A new study created in memory with name: no-name-f83deed5-a57c-4dfa-9c4f-a5f92db796b0


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:31:28,690] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:31:41,005] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:37:48,880] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.6963599701922215.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:38:28,094] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:47:30,163] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.6963599701922215.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:47:53,902] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 22:14:21,972] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 22 with value: 0.23482487516663234.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:15:00,745] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.0985092209

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:21:06,294] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8952643616873973, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.6509686174553241, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 'auto', 'min_impurity_decrease': 2.2946753237767036e-06, 'validation_fraction': 0.8670144195682054, 'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 18}. Best is trial 22 with value: 0.23482487516663234.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:21:23,664] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9559398584951578, 'learning_rate': 0.0013120255

Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 22:27:19,090] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9040601608260395, 'learning_rate': 0.008028762030775153, 'dropout_rate': 0.1661055390191164, 'n_estimators': 412, 'criterion': 'squared_error', 'ccp_alpha': 0.5920167940309409, 'min_weight_fraction_leaf': 0.20760648939509768, 'max_features': 1, 'min_impurity_decrease': 1.2858411523836384e-06, 'validation_fraction': 0.7519570151291436, 'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 22 with value: 0.23482487516663234.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:35:48,148] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.003888190949209

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:46:29,889] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8783477148356468, 'learning_rate': 0.019145642246983192, 'dropout_rate': 0.2500967923284591, 'n_estimators': 479, 'criterion': 'squared_error', 'ccp_alpha': 1.2742275973203492, 'min_weight_fraction_leaf': 0.11068414611763369, 'max_features': 'auto', 'min_impurity_decrease': 0.0011701047770450368, 'validation_fraction': 0.9552938036430088, 'min_samples_split': 14, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 4}. Best is trial 85 with value: 0.23333641670051097.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 23:03:59,685] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9225585795675643, 'learning_rate': 0.0028206049

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 23:09:22,504] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9084360842440081, 'learning_rate': 0.013060448266216875, 'dropout_rate': 0.25443633448028735, 'n_estimators': 468, 'criterion': 'squared_error', 'ccp_alpha': 0.2223066295281712, 'min_weight_fraction_leaf': 0.25005462596245476, 'max_features': 'auto', 'min_impurity_decrease': 2.1305992007975229e-07, 'validation_fraction': 0.4189919199463679, 'min_samples_split': 15, 'max_leaf_nodes': 12, 'min_samples_leaf': 19, 'max_depth': 1}. Best is trial 85 with value: 0.23333641670051097.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.23333641670051097], datetime_start=datetime.datetime(2024, 4, 16, 22, 43, 46, 766968), datetime_complete=datetime.datetime(2024, 4, 16, 22, 44, 27, 130503), params={'subsample': 0.9481727373988897

In [71]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [72]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.73
train_ibs:  0.233


#### Test

In [73]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [74]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03525202624769574,
                                 dropout_rate=0.9694253886674298,
                                 learning_rate=0.05377375104309677, max_depth=4,
                                 max_features='log2', max_leaf_nodes=15,
                                 min_impurity_decrease=2.482390348219561e-06,
                                 min_samples_leaf=15, min_samples_split=19,
                                 min_weight_fraction_leaf=0.3666298089106992,
                                 n_estimators=450, random_state=123,
                                 subsample=0.9035029991499002,
                                 validation_fraction=0.8472602359999014)

C-index score: 0.591


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01138981446692314,
                                 criterion='squared_error',
                                 dropout_rate=0.2479619862207481,
                                 learning_rate=0.02022975259096714, max_depth=8,
                                 max_features='auto', max_leaf_nodes=20,
                                 min_impurity_decrease=9.61920586779085e-07,
                                 min_samples_leaf=18, min_samples_split=14,
                                 min_weight_fraction_leaf=0.026767353450782638,
                                 n_estimators=438, random_state=123,
                                 subsample=0.9481727373988897,
                                 validation_fraction=0.973754058089352)

IBS: 0.229


In [75]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [76]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [77]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 23:09:32,717] A new study created in memory with name: no-name-e22d06e7-b2e1-4a19-a4ca-855f553d5075


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6609442060085837
[I 2024-04-16 23:09:33,594] Trial 0 finished with value: 0.6249682148055231 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6566523605150214
[I 2024-04-16 23:09:39,339] Trial 1 finished with value: 0.6241098457068107 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.574468085106383
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.6781115879828327
[I 2024-04-16 23:10:24,751] Trial 19 finished with value: 0.6456442843661956 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 11 with value: 0.6592329087370825.
Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6738197424892703
[I 2024-04-16 23:10:29,606] Trial 20 finished with value: 0.6399513549568285 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 11 with value: 0.6592329087370825.
Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.5787234042553191
Fold 4 C-index: 0.6920152091254753


Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6652360515021459
[I 2024-04-16 23:11:26,119] Trial 38 finished with value: 0.6280863337749738 and parameters: {'subsample': 0.3297325755614151, 'dropout_rate': 0.8867789132968811, 'n_estimators': 392, 'learning_rate': 0.08480908032505553}. Best is trial 11 with value: 0.6592329087370825.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6781115879828327
[I 2024-04-16 23:11:27,051] Trial 39 finished with value: 0.6408313430060953 and parameters: {'subsample': 0.1910828860959901, 'dropout_rate': 0.29848665859072016, 'n_estimators': 145, 'learning_rate': 0.06892741183938003}. Best is trial 11 with value: 0.6592329087370825.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745


...

Fold 4 IBS: 0.2426706185564074
Fold 5 IBS: 0.22283081379821795
[I 2024-04-16 23:13:41,906] Trial 19 finished with value: 0.24310221296462103 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.24550608887605468
Fold 2 IBS: 0.2133505584021398
Fold 3 IBS: 0.23294747738469943
Fold 4 IBS: 0.23024731593258208
Fold 5 IBS: 0.2123372739360029
[I 2024-04-16 23:13:42,321] Trial 20 finished with value: 0.22687774290629575 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2448259892509062
Fold 2 IBS: 0.21786672573833757
Fold 3 IBS: 0.23050391968540718
Fold 4 IBS: 0.23249100529840805
Fold 5 IBS: 0.215533050087791
[I 2024-04-16 23:13:42,661] Trial 21 finished with value: 0.2282441380121

Fold 5 IBS: 0.2122914778844625
[I 2024-04-16 23:13:56,944] Trial 38 finished with value: 0.2333975796328732 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2943463281154152
Fold 2 IBS: 0.21435275898375683
Fold 3 IBS: 0.292776623523439
Fold 4 IBS: 0.2606234117164125
Fold 5 IBS: 0.24191548716779984
[I 2024-04-16 23:13:57,274] Trial 39 finished with value: 0.26080292190136467 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2956705905294083
Fold 2 IBS: 0.21430445848659016
Fold 3 IBS: 0.2928588854990794
Fold 4 IBS: 0.2596622123064148
Fold 5 IBS: 0.24678618384472736
[I 2024-04-16 23:13:57,904] Trial 40 finished with value: 0.261856466133244 and parameters: {'subsample': 0.8

Fold 3 IBS: 0.24466604990129345
Fold 4 IBS: 0.22951388023914565
Fold 5 IBS: 0.21232254648299242
[I 2024-04-16 23:14:07,570] Trial 58 finished with value: 0.22873557075266032 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2450766728615442
Fold 2 IBS: 0.21570571438138603
Fold 3 IBS: 0.23170031253784298
Fold 4 IBS: 0.23100841297079955
Fold 5 IBS: 0.21387190006582926
[I 2024-04-16 23:14:07,783] Trial 59 finished with value: 0.2274726025634804 and parameters: {'subsample': 0.7169412850975417, 'dropout_rate': 0.7863461136932807, 'n_estimators': 20, 'learning_rate': 0.03702429542217137}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2799925625848654
Fold 2 IBS: 0.20541721127231735
Fold 3 IBS: 0.27641891339421315
Fold 4 IBS: 0.24728738758668545
Fold 5 IBS: 0.231501204035336
[I 2024-04-16 23:14:08,391] Trial 60 finish

Fold 3 IBS: 0.2432701864482772
Fold 4 IBS: 0.2284683602008423
Fold 5 IBS: 0.21006566848816424
[I 2024-04-16 23:14:24,856] Trial 77 finished with value: 0.22756786578412544 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.24728609182637104
Fold 2 IBS: 0.20948456237788454
Fold 3 IBS: 0.2367879689804507
Fold 4 IBS: 0.22900068533758589
Fold 5 IBS: 0.21042389725248523
[I 2024-04-16 23:14:25,273] Trial 78 finished with value: 0.2265966411549555 and parameters: {'subsample': 0.5939500427136554, 'dropout_rate': 0.7406477151463077, 'n_estimators': 73, 'learning_rate': 0.015420072941943445}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2491522359439618
Fold 2 IBS: 0.20720785646863638
Fold 3 IBS: 0.23970965460068624
Fold 4 IBS: 0.22880108027481874
Fold 5 IBS: 0.2102336161969871
[I 2024-04-16 23:14:25,752] Trial 79 fin

Fold 4 IBS: 0.2978922662575483
Fold 5 IBS: 0.3041795583962892
[I 2024-04-16 23:14:33,535] Trial 96 finished with value: 0.31517286801080535 and parameters: {'subsample': 0.41313939490511475, 'dropout_rate': 0.5213109482382285, 'n_estimators': 158, 'learning_rate': 0.054487916689644846}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.24708172105946397
Fold 2 IBS: 0.21005062605084635
Fold 3 IBS: 0.23658239945020892
Fold 4 IBS: 0.2291089640988238
Fold 5 IBS: 0.2105403389046395
[I 2024-04-16 23:14:33,968] Trial 97 finished with value: 0.2266728099127965 and parameters: {'subsample': 0.5389995281019297, 'dropout_rate': 0.6077476353900257, 'n_estimators': 73, 'learning_rate': 0.014931416501109941}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.26183286577938214
Fold 2 IBS: 0.20183342233582774
Fold 3 IBS: 0.2575253907596287
Fold 4 IBS: 0.23417362585495322
Fold 5 IBS: 0.21506795896798186
[I 2024-04-16 23:14:34,890] Trial 98 finished with value: 0.23408665273955

In [78]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [79]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.659
train_ibs:  0.227


#### Test

In [80]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [81]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10580803679784617,
                                              learning_rate=0.08933703080421439,
                                              n_estimators=324,
                                              random_state=123,
                                              subsample=0.11557957726835853)

C-index score: 0.536


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6341528845468055,
                                              learning_rate=0.014291926646619518,
                                              n_estimators=76, random_state=123,
                                              subsample=0.6070545538128025)

IBS: 0.234


In [82]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [83]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.835,1.0
Randomsurvivalforest,0.805,2.0
GradientBoosting,0.730,3.0
CoxPH,0.710,4.0
CoxLasso,0.709,5.5
CoxElastic,0.709,5.5
ComponentwiseGradientBoosting,0.659,7.0
CoxRidge,0.655,8.0


In [84]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.196,1.0
ExtraSurvivalTrees,0.198,2.0
CoxLasso,0.200,3.5
CoxElastic,0.200,3.5
CoxPH,0.201,5.0
ComponentwiseGradientBoosting,0.227,6.0
GradientBoosting,0.233,7.0
CoxRidge,0.236,8.0


In [85]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.591,1.0
ExtraSurvivalTrees,0.581,2.0
Randomsurvivalforest,0.579,3.0
CoxLasso,0.566,4.0
CoxPH,0.565,5.5
CoxElastic,0.565,5.5
CoxRidge,0.537,7.0
ComponentwiseGradientBoosting,0.536,8.0


In [86]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
CoxRidge,0.229,1.5
GradientBoosting,0.229,1.5
ExtraSurvivalTrees,0.234,3.5
ComponentwiseGradientBoosting,0.234,3.5
Randomsurvivalforest,0.247,5.0
CoxLasso,0.264,6.5
CoxElastic,0.264,6.5
CoxPH,0.265,8.0


In [87]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/dfs/standard/plsr/'  # Folder path where you want to save the files

# List of corresponding file namess
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_dfs_standard_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [88]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-16
